In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import VectorStoreIndex
from llama_index.llms.ollama import Ollama
from llama_index.core.embeddings import resolve_embed_model
from dotenv import load_dotenv
# from llama_index.core.response.notebook_utils import display_source_node
import os
from google import genai

load_dotenv()
reader = SimpleDirectoryReader(os.path.join("../", "data"))
documents = reader.load_data()

In [ ]:
node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=20)

In [ ]:
base_nodes = node_parser.get_nodes_from_documents(documents)
# set node ids to be a constant
for idx, node in enumerate(base_nodes):
    node.id_ = f"node-{idx}"

In [ ]:
embed_model = "text-embedding-3-small"

embed_model_instance = resolve_embed_model("local:BAAI/bge-small-en")
base_index = VectorStoreIndex(base_nodes, embed_model=embed_model_instance)
base_index

In [ ]:
base_retriever = base_index.as_retriever(similarity_top_k=4)

In [ ]:
query = "What is my PNR number for the flight Mumbai to New Delhi and What date is my flight booked for?"

In [ ]:
retrievals = base_retriever.retrieve(
    query
)

In [ ]:
for n in retrievals:
    print(n)


In [ ]:
gemini_api_key = os.getenv("GEMINI_API_KEY")

In [ ]:
PROMPT_PATH = r"prompt_llama_index.yaml"

def load_prompt(path=PROMPT_PATH):
    """Load prompt config strictly from a YAML file."""
    from pathlib import Path
    import yaml  # pip install pyyaml

    text = Path(path).read_text(encoding="utf-8")
    cfg = yaml.safe_load(text) or {}
    
    return cfg

In [ ]:
client = genai.Client(api_key=gemini_api_key)

prompt = load_prompt()

final_prompt = prompt["template"].format(
    context="\n".join([n.get_content() for n in retrievals]),
    question=query
)

response = client.models.generate_content(
    model="gemini-2.5-flash", contents=final_prompt
)

print(response.text)